In [45]:
%run cochain_complex.ipynb
m=8

In [46]:
g=Symp_symb(2*m+1)
W=g.ext_alg
C=g.cochain_complex
y,h,e=symbols('y,h,e')
K=IndexedBase('K')

Y,H,E,X=g.basis[0:4]
for i in range(1,2*m+1):
    exec(f'e{i}=g.basis[{i+3}]')
N=g.basis[-1]

omega=W.elt({})
for i in range(1,(m+1)):
    omega+=(-1)**(i+1)*g.basis[i+3].cast_as_ext_elt().wedge(g.basis[-1-i].cast_as_ext_elt())


In [47]:
def nonWilc_proj(ext_elt):
    # E^{0,2}_2 from the first Lefschetz term
    r=ext_elt.parent.elt(copy.deepcopy(ext_elt.vd))
    for d in ext_elt.vd:
        for w in ext_elt.vd[d]:
            for i in range(len(ext_elt.vd[d][w])): 
                for a in ext_elt.parent.basis(d,w)[i].components[0:-1]:
                    if str(a)[0]!='e':
                        try: r.vd[d][w][i]=0
                        except:
                            # Maybe w isn't in r.vd[d]
                            r.vd[d][w]=Matrix(r.vd[d][w])
                            r.vd[d][w][i]=0
    r.clear_zeros()
    return r

def Wilc_proj(c):
    # E^{1,1}_2 from the first Lefschetz term
    r=c.parent.elt(copy.deepcopy(c.vd))
    for d in c.vd:
        for w in c.vd[d]:
            for i in range(len(c.vd[d][w])):
                a=c.parent.basis(d,w)[i].components
                if a[0]!=X:
                    r.vd[d][w][i]=0
                else:
                    for j in range(1,len(a)-1):
                        if str(a[j])[0] not in ['e']:
                            r.vd[d][w][i]=0
    r.clear_zeros()
    return r

def psi(ext_elt):
    r=C.elt({})
    X_cache=[nonWilc_proj(ext_elt)]
    for i in range(2*m):
        r+=X_cache[i].tensor(g.basis[2*m-i+3])
        X_cache.append((-X).ad(X_cache[-1],mod='m_dual'))
    return r

def eta_proj(c):
    # This is the second term from the Lefschetz proposition
    r=c.parent.elt(copy.deepcopy(c.vd))
    for d in c.vd:
        for w in c.vd[d]:
            for i in range(len(c.vd[d][w])):
                a=c.parent.basis(d,w)[i].components
                if a[-2]!=N:
                    r.vd[d][w][i]=0
    r.clear_zeros()
    return r

def C_V_gl2_proj(c):
    # For Spencer operator
    r=c.parent.elt(copy.deepcopy(c.vd))
    for d in c.vd:
        for w in c.vd[d]:
            for i in range(len(c.vd[d][w])):
                a=c.parent.basis(d,w)[i].components
                for j in range(len(a)-1):
                    if str(a[j])[0]!='e':
                        r.vd[d][w][i]=0
                if (str(a[-1]) not in ['Y','H','E','X']):
                    r.vd[d][w][i]=0
    r.clear_zeros()
    return r

def Spencer_op_on_base(c):
    """Assuming c is an elt of C(V,gl2), computes its Spencer image"""
    r=C.elt({})
    v=W.elt_from_cd({tuple([str(a) for a in c.components[0:-1]]):1})
    for a in g.basis[4:-1]:
        t=c.components[-1].ad(a)
        r+=-a.cast_as_ext_elt().wedge(v).tensor(t)
    return r

def Spencer_op(c):
    new_c=C_V_gl2_proj(c)
    r=C.elt({})
    for d in new_c.vd:
        for w in new_c.vd[d]:
            for i in range(len(new_c.vd[d][w])):
                if new_c.vd[d][w][i]!=0:
                    r+=new_c.vd[d][w][i]*Spencer_op_on_base(C.basis(d,w)[i])
    return r

def Lefschetz_op(ext_elt):
    return omega.wedge(ext_elt)

def Q_map(ext_elt,Q_ker_basis):
    "Note: Q_ker_basis should be orthonormal"
    r=W.elt(copy.deepcopy(ext_elt.vd))
    for b in Q_ker_basis:
        coeff=-W.iprod(b,ext_elt)/W.iprod(b,b)
        r+=coeff*b
    return r

def tensor_solve(t):
    eqs=[]
    for d in t.vd:
        for w in t.vd[d]:
            for i in range(len(t.vd[d][w])):
                if t.vd[d][w][i]!=0: eqs.append(t.vd[d][w][i])
    return solve(eqs)


In [48]:
a1=C.elt_from_cd({('e_1','E'):1})
a2=C.elt_from_cd({('e_1','X'):1})
a3=C.elt_from_cd({('e_2','X'):2,('e_1','H'):-1})
a4=C.elt_from_cd({('e_3','X'):2,('e_2','H'):-1,('e_1','Y'):-1})

b1=W.elt_from_cd({('e_1',f'e_{2*m}'):1})
b2=W.elt_from_cd({('e_1',f'e_{2*m-1}'):1})
b3=W.elt_from_cd({('e_2',f'e_{2*m-1}'):2,('e_1',f'e_{2*m}'):-(2*m-1)})
b4=W.elt_from_cd({('e_3',f'e_{2*m-1}'):2,('e_2',f'e_{2*m}'):-(2*m-1)})

In [49]:
# # Double checking the quotient elements

# # psi(L_b) is what we should quotient by to obtain H^2(g_-,g) from ker(X^{2m}|_{V^{wedge 2}})
# L_a=[a1,a2,a3,a4]
# L_b=[b1,b2,b3,b4]
# for i in range(4):
#     if psi(L_b[i])-Spencer_op(L_a[i])!=C.elt({}): print('Failure at',i)

# # [psi(omega)] is the quotient from the Lefschetz theorem
# if psi(omega)-Lefschetz_op(eval(f'e{2*m}.cast_as_cochain()'))!=C.elt({}):
#     print('psi(omega)!=Lefschetz(e_{2m})')

# # This kernel should be in the kernel of the connecting boundary map
# for b in L_b+[omega]:
#     if C.subspace_proj(psi(b).cb(),'coclosed')!=C.elt({}):
#         print('Failure: connecting map,',b)

# # This kernel should be in Inv_X
# for b in L_b+[omega]:
#     if X.ad(psi(b))!=C.elt({}):
#         print('Failure: Inv_X test,',b)

# # Check: does psi have weight -2m? 
# for w in W.basis(2):
#     for b in W.basis(2,w):
#         im_elt=psi(b)
#         im_elt.clear_zeros()
#         if 2 in im_elt.vd:
#             for k in im_elt.vd[2]:
#                 if k!=b.wght-2*m: print('Failure: weight test',b)


In [50]:
# Are all elts in the image of psi X-invariant? No! Indeed, this is a mistake in Medvedev's preprint
for w in W.basis(2):
    for a in W.basis(2,w):
        if X.ad(psi(a))!=W.elt({}): print('psi',a,'is not X-invariant')

psi (e_3,e_16) is not X-invariant
psi (e_4,e_15) is not X-invariant
psi (e_5,e_14) is not X-invariant
psi (e_6,e_13) is not X-invariant
psi (e_7,e_12) is not X-invariant
psi (e_8,e_11) is not X-invariant
psi (e_9,e_10) is not X-invariant
psi (e_4,e_16) is not X-invariant
psi (e_5,e_15) is not X-invariant
psi (e_6,e_14) is not X-invariant
psi (e_7,e_13) is not X-invariant
psi (e_8,e_12) is not X-invariant
psi (e_9,e_11) is not X-invariant
psi (e_5,e_16) is not X-invariant
psi (e_6,e_15) is not X-invariant
psi (e_7,e_14) is not X-invariant
psi (e_8,e_13) is not X-invariant
psi (e_9,e_12) is not X-invariant
psi (e_10,e_11) is not X-invariant
psi (e_6,e_16) is not X-invariant
psi (e_7,e_15) is not X-invariant
psi (e_8,e_14) is not X-invariant
psi (e_9,e_13) is not X-invariant
psi (e_10,e_12) is not X-invariant
psi (e_7,e_16) is not X-invariant
psi (e_8,e_15) is not X-invariant
psi (e_9,e_14) is not X-invariant
psi (e_10,e_13) is not X-invariant
psi (e_11,e_12) is not X-invariant
psi (e_8,e

In [51]:
# Construct an arbitrary element of Wedge^2(V^*) of weight >=2*m+1
# Since psi has wght -2*m, these are still taken to the positive cochain complex
alpha=IndexedBase('alpha')
a=W.elt({})
for w in W.basis(2):
    for b in W.basis(2,w):
        if b.wght>2*m:
            i,j=[g.basis.index(b.components[s])-3 for s in [0,1]]
            if i in range(1,2*m+1) and j in range(1,2*m+1):
                new_elt=W.elt(copy.deepcopy(b.vd))
                for d in b.vd:
                    for w in b.vd[d]:
                        for k in range(len(b.vd[d][w])):
                            if new_elt.vd[d][w][k]==1: new_elt.vd[d][w][k]=alpha[i,j]
                a+=new_elt

# Solve X**2m(a)=0, which will make psi(a) X-invariant
X2ma=copy.deepcopy(a)
for i in range(2*m):
    X2ma=(-X).ad(X2ma,mod='E')
s0=tensor_solve(X2ma) # this makes psi(a) X-invariant
a_X_invar=a.subs(s0)
simplify_cochain(a_X_invar)

# We want to make the coboundary zero in cohomology
# Note: C_+^1(g,R*N) is zero, so the exact 2-forms are precisely zero; no need to quotient
s1=tensor_solve(psi(a_X_invar).cb())

# Checking the above solving
if X.ad(psi(a.subs(s0)))!=C.elt({}): print('Failed test 1')
if psi(a.subs(s0).subs(s1)).cb()!=C.elt({}): print('Failed test 2')

a_X_invar_closed=a_X_invar.subs(s1) #
simplify_cochain(a_X_invar_closed)

In [52]:
a_X_invar

alpha[1, 16]*(e_1,e_16)+alpha[2, 15]*(e_2,e_15)+alpha[3, 14]*(e_3,e_14)+alpha[4, 13]*(e_4,e_13)+alpha[5, 12]*(e_5,e_12)+alpha[6, 11]*(e_6,e_11)+alpha[7, 10]*(e_7,e_10)+alpha[8, 9]*(e_8,e_9)+alpha[2, 16]*(e_2,e_16)+alpha[3, 15]*(e_3,e_15)+alpha[4, 14]*(e_4,e_14)+alpha[5, 13]*(e_5,e_13)+alpha[6, 12]*(e_6,e_12)+alpha[7, 11]*(e_7,e_11)+alpha[8, 10]*(e_8,e_10)+(-55*alpha[4, 15]/13 - 315*alpha[5, 14]/26 - 49*alpha[6, 13]/2 - 35*alpha[7, 12] - 33*alpha[8, 11] - 55*alpha[9, 10]/4)*(e_3,e_16)+alpha[4, 15]*(e_4,e_15)+alpha[5, 14]*(e_5,e_14)+alpha[6, 13]*(e_6,e_13)+alpha[7, 12]*(e_7,e_12)+alpha[8, 11]*(e_8,e_11)+alpha[9, 10]*(e_9,e_10)+(-25*alpha[5, 15]/8 - 7*alpha[6, 14] - 91*alpha[7, 13]/8 - 13*alpha[8, 12] - 143*alpha[9, 11]/16)*(e_4,e_16)+alpha[5, 15]*(e_5,e_15)+alpha[6, 14]*(e_6,e_14)+alpha[7, 13]*(e_7,e_13)+alpha[8, 12]*(e_8,e_12)+alpha[9, 11]*(e_9,e_11)+(-18*alpha[6, 15]/11 - 98*alpha[7, 14]/55 - 13*alpha[8, 13]/11 - 39*alpha[9, 12]/110)*(e_5,e_16)+alpha[6, 15]*(e_6,e_15)+alpha[7, 14]*(e_7

In [53]:
X_invar_basis=[]
for A in Indexed_obj_in_expr(a_X_invar.symb_expr()):
    sd={B:0 for B in Indexed_obj_in_expr(a_X_invar.symb_expr())}
    sd[A]=1
    X_invar_basis.append(a_X_invar.subs(sd))
for a in X_invar_basis:
    a.clear_zeros()

temp={}
for A in X_invar_basis:
    w=list(A.vd[2].keys())[0]
    if w not in temp: temp[w]=[]
    temp[w].append(A)

temp

{19: [-55/4*(e_3,e_16)+(e_9,e_10),
  -315/26*(e_3,e_16)+(e_5,e_14),
  -35*(e_3,e_16)+(e_7,e_12),
  -55/13*(e_3,e_16)+(e_4,e_15),
  -49/2*(e_3,e_16)+(e_6,e_13),
  -33*(e_3,e_16)+(e_8,e_11)],
 21: [-18/11*(e_5,e_16)+(e_6,e_15)-54/143*(e_10,e_11),
  -39/110*(e_5,e_16)+(e_9,e_12)-126/55*(e_10,e_11),
  -98/55*(e_5,e_16)+(e_7,e_14)-882/715*(e_10,e_11),
  -13/11*(e_5,e_16)+(e_8,e_13)-24/11*(e_10,e_11)],
 20: [-143/16*(e_4,e_16)+(e_9,e_11),
  -13*(e_4,e_16)+(e_8,e_12),
  -25/8*(e_4,e_16)+(e_5,e_15),
  -7*(e_4,e_16)+(e_6,e_14),
  -91/8*(e_4,e_16)+(e_7,e_13)],
 17: [(e_5,e_12),
  (e_3,e_14),
  (e_4,e_13),
  (e_2,e_15),
  (e_7,e_10),
  (e_8,e_9),
  (e_6,e_11),
  (e_1,e_16)],
 23: [-5/36*(e_7,e_16)+(e_9,e_14)-35/13*(e_10,e_13)+105/26*(e_11,e_12),
  -5/9*(e_7,e_16)+(e_8,e_15)-32/13*(e_10,e_13)+60/13*(e_11,e_12)],
 18: [(e_2,e_16),
  (e_4,e_14),
  (e_5,e_13),
  (e_3,e_15),
  (e_6,e_12),
  (e_7,e_11),
  (e_8,e_10)],
 22: [-13/32*(e_6,e_16)+(e_9,e_13)-21/16*(e_10,e_12),
  -5/4*(e_6,e_16)+(e_7,e_15)-9/

In [54]:
# Gram-Schmidt process for the basis of the quotient
U_basis=[W.elt(copy.copy(s.vd)) for s in [b1,b2,b3,b4,omega]]
for i in range(len(U_basis)):
    for j in range(i):
        U_basis[i]+=-W.iprod(U_basis[i],U_basis[j])/W.iprod(U_basis[j],U_basis[j])*U_basis[j]
    U_basis[i]=U_basis[i]*sqrt(Rational(1,W.iprod(U_basis[i],U_basis[i])))

# Checking Gram-Schmidt
for i in range(len(U_basis)):
    for j in range(len(U_basis)):
        temp=W.iprod(U_basis[i],U_basis[j])
        if temp!=0 and i!=j: print('Failure:',i,j,'-->',temp)
        if temp!=1 and i==j: print('Failure:',i,j,'-->',temp)

In [55]:
# Substitute so that the projection is a bijection
# This corresponds to choosing a subspace of Inv_X(Hom(V\wedge V,V)) \cap ker(d)
temp=tensor_solve(Q_map(a_X_invar_closed,U_basis))
s2=tensor_solve(a_X_invar_closed.subs(temp))
a_proj=a_X_invar_closed.subs(s2)
simplify_cochain(a_proj)

# This assures we didn't change the image
if Q_map(a_X_invar_closed.subs(temp),U_basis)!=C.elt({}): print('image changed')

# Check for injectivity
# Same Indexed objects involved
temp=Q_map(a_proj,U_basis)
simplify_cochain(temp)
if not Indexed_obj_in_expr(a_proj)==Indexed_obj_in_expr(temp): print('Q_map is not injective')

# No redundancy in indexed objects
for k in tensor_solve(a_proj):
    if tensor_solve(a_proj)[k]!=0: print('Q_map is not injective')

In [56]:
# temp=C.subspace_proj(psi(a_proj),'harmonic')
# simplify_cochain(temp)
# factor(temp.vd)

In [57]:
A_basis=[]
for A in Indexed_obj_in_expr(a_proj.symb_expr()):
    sd={B:0 for B in Indexed_obj_in_expr(a_proj.symb_expr())}
    sd[A]=1
    A_basis.append(a_proj.subs(sd))

In [58]:
wght_list=[]
for A in A_basis:
    temp=psi(A)
    temp.clear_zeros()
    L=list(temp.vd[2].keys())
    if len(L)!=1:
        print('multi-weight basis element',A)
    else: wght_list.append(L[0])
wght_list.sort()
print(wght_list)

[1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 8]


In [59]:
len(A_basis)

21

In [60]:
# How many nonWilc invars for a (2,n) distribution?
# I should check the weights for each of these

# m ----> n ---> #(NwI) ---> weights
# 3 ----> 6 ----> 1 -------> (3)
# 4 ----> 7 ----> 4 -------> (1,2,3,4)
# 5 ----> 8 ----> 7 -------> (1,2,2,3,3,4,5)
# 6 ----> 9 ----> 11 ------> (1,1,2,2,3,3,3,4,4,5,6)
# 7 ----> 10 ---> 16 ------> ((1)x3,(2)x3,(3)x3,(4)x3,(5)x2,6,7)
# 8 ----> 11 ---> 21 ------> ((1)x3,(2)x4,(3)x4,(4)x3,(5)x3,(6)x2,7,8)
# 9 ----> 12 ---> 27 ------> 
# 10 ---> 13 ---> 34 ------> 

## Checking Manual Computations

For $F = \sum_{i+j>2m}F^{ij}\varepsilon_i^*\wedge\varepsilon_j^*$ with $F^{ij}=-F^{ji}$, we have that $X.\psi(F)=0$ if and only if $(-X)^{2m}.F=0$, which is equivalent to
$$
    \sum_{a=1}^{2m}\sum_{b=a+1}^{2m}\sum_{i=a_b}^{\lfloor\frac{a+b+2m}{2}\rfloor} 2\Big(\binom{2m}{i-a}-\binom{2m}{i-b}\Big)F^{i,(a+b+2m-i)}\varepsilon_a^*\wedge\varepsilon_b^* = 0
$$ 

In [61]:
F=IndexedBase('F')

def binom(n,m):
    try: return binomial_coefficients(n)[(m,n-m)]
    except: return 0

In [62]:
eqs=[]
eqs_dict={}
for c in range(1,2*m+1):
    for b in range(c+1,2*m+1):
        eq=0
        for i in range(c+b,2*m+1):
            j=c+b+2*m-i
            if j in range(1,2*m+1):
                if i<j: eq+=(binom(2*m,i-c)-binom(2*m, i-b))*F[i,j]
                elif j<i: eq+=(binom(2*m,i-c)-binom(2*m, i-b))*(-F[j,i])
        eqs.append(eq)
        eqs_dict[(c,b)]=eq
s=solve(eqs)

In [63]:
F2ma=X2ma.subs({alpha:F})
if F2ma.subs(s)!=W.elt({}):
    print('Manual solving incorrect (1)')
if set(s.keys())!=set([k.subs({alpha:F}) for k in s0]): 
    print('Manual solving incorrect (2)')

In [64]:
list(s.keys())

[F[10, 11],
 F[10, 12],
 F[10, 13],
 F[10, 14],
 F[10, 15],
 F[10, 16],
 F[11, 12],
 F[11, 13],
 F[11, 14],
 F[11, 15],
 F[11, 16],
 F[12, 13],
 F[12, 14],
 F[12, 15],
 F[12, 16],
 F[13, 14],
 F[13, 15],
 F[13, 16],
 F[14, 15],
 F[14, 16],
 F[15, 16],
 F[3, 16],
 F[4, 16],
 F[5, 16],
 F[6, 16],
 F[7, 16],
 F[8, 16],
 F[9, 16]]

## Toying around

In [65]:
for w in C.basis(2):
    for c in C.basis(2,w):
        if Wilc_proj(c)!=C.elt({}):
            elt=Wilc_proj(c).cb()
            print(Wilc_proj(c))
            print(Wilc_proj(elt),'\n')
            #if Wilc_proj(elt)+eta_proj(elt)!=elt: print(c,'\n') # Wilc is taken to Wilc+eta part

(X,e_1,Y)
-15*(X,e_1,e_2,e_1)-28*(X,e_1,e_3,e_2)-39*(X,e_1,e_4,e_3)-48*(X,e_1,e_5,e_4)-55*(X,e_1,e_6,e_5)-60*(X,e_1,e_7,e_6)-63*(X,e_1,e_8,e_7)-64*(X,e_1,e_9,e_8)-63*(X,e_1,e_10,e_9)-60*(X,e_1,e_11,e_10)-55*(X,e_1,e_12,e_11)-48*(X,e_1,e_13,e_12)-39*(X,e_1,e_14,e_13)-28*(X,e_1,e_15,e_14)-15*(X,e_1,e_16,e_15) 

(X,e_2,H)
-15*(X,e_1,e_2,e_1)+11*(X,e_2,e_3,e_3)+9*(X,e_2,e_4,e_4)+7*(X,e_2,e_5,e_5)+5*(X,e_2,e_6,e_6)+3*(X,e_2,e_7,e_7)+(X,e_2,e_8,e_8)-(X,e_2,e_9,e_9)-3*(X,e_2,e_10,e_10)-5*(X,e_2,e_11,e_11)-7*(X,e_2,e_12,e_12)-9*(X,e_2,e_13,e_13)-11*(X,e_2,e_14,e_14)-13*(X,e_2,e_15,e_15)-15*(X,e_2,e_16,e_16) 



KeyboardInterrupt: 

In [ ]:
# It looks like little can be said about coboundary(NonWilc)
for w in C.basis(2):
    for c in C.basis(2,w):
        if nonWilc_proj(c).cb()!=C.elt({}): 
            print(nonWilc_proj(c))
            print(nonWilc_proj(c).cb(),'\n')

In [27]:
arb_c=C.elt({})
for i in range(len(g.m_basis)):
    for j in range(i+1,len(g.m_basis)):
        for k in range(len(g.basis)):
            d,w,p=C.dwi((g.m_basis_strs[i],g.m_basis_strs[j],g.basis_strs[k]))
            M=zeros(len(C.basis(d,w)),1)
            M[p]=IndexedBase('c')[i+3,j+3,k]
            arb_c+=C.elt({d:{w:M}})

In [ ]:
v=nonWilc_proj(arb_c)
vv=Gerst_prod(v,v)

w=Wilc_proj(arb_c)
ww=Gerst_prod(w,w)

eta_part=eta_proj(arb_c)
eta_eta=Gerst_prod(eta_part,eta_part)

print(Wilc_proj(ww)==ww) # the Gerst square preserves Wilc, nonWilc, and eta parts
print(nonWilc_proj(vv)==vv) # the Gerst square preserves the nonWilc part
print(eta_proj(eta_eta)==eta_eta)

In [ ]:
wv=Gerst_prod(w,v)
vw=Gerst_prod(v,w)
print(Wilc_proj(wv)==wv) # wv is mixed
print(Wilc_proj(vw)==vw) # but wv is in the Wilc part

In [ ]:
eta_v=Gerst_prod(eta_part,v)
v_eta=Gerst_prod(v,eta_part)
print(eta_proj(eta_v)==eta_v) # eta_v is mixed
print(eta_proj(v_eta)==v_eta) # v_eta is in the eta part

In [ ]:
eta_w=Gerst_prod(eta_part,w)
w_eta=Gerst_prod(w,eta_part)
print(eta_proj(eta_w)+Wilc_proj(eta_w)==eta_w) # eta_w is in eta part plus Wilc part
print(eta_proj(w_eta)==w_eta) # w_eta is in the eta part

In [ ]:
for w in C.basis(3):
    for i in range(len(C.basis(3,w))):
        try: 
            if ww.vd[3][w][i]!=0: print(C.basis(3,w)[i])
        except: pass

In [29]:
c_harm=C.subspace_proj(arb_c,'harmonic')
c_coexact=C.subspace_proj(arb_c,'coexact')

In [30]:
ch_ch=Gerst_prod(c_harm,c_harm)

In [ ]:
# > 1 hr for m=4
ch_ch_h=C.subspace_proj(ch_ch,'harmonic')

In [ ]:
ch_ch_ce=C.subspace_proj(ch_ch,'coexact')

In [ ]:
ch_ch_e=C.subspace_proj(ch_ch,'exact')